## Match Top Monthly Books by Checkout Volume to Ebook Checkouts to see if popularity persists cross-channel

### Setup data

In [24]:
## Import packages and datasets
import glob
import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
import numpy as np
from rapidfuzz import process, fuzz
import re

processed_dir = '/Users/audriswong/data-portfolio/projects/seattle-checkouts/data/processed'

book_checkouts = pd.read_parquet(f'{processed_dir}/book_checkouts.parquet').rename(columns={'title_clean': 'title'}) 
ebook_checkouts = pd.read_parquet(f'{processed_dir}/ebook_checkouts.parquet').rename(columns={'title_clean': 'title'})

In [3]:
#rank add monthly checkout ranking per work to all titles

def works_with_monthly_rank(df, year_col='checkoutyear', month_col='checkoutmonth'):
    results = []
    for (year, month), group in df.groupby([year_col, month_col]):
        ranked = (group.groupby('work_key')
                .agg(total_checkouts=('checkouts', 'sum'),
                     genre=('genre', 'first'),
                     title=('title', 'first'),
                     creator=('creator_clean', 'first'),
                     month_date = ('month_date', 'first'))
                .reset_index()
                .sort_values('total_checkouts', ascending=False))
        ranked['month_checkout_rank'] = range(1, len(ranked) + 1)
        ranked['checkoutyear']  = year
        ranked['checkoutmonth'] = month
        results.append(ranked)
    return pd.concat(results, ignore_index=True)

book_ranked  = works_with_monthly_rank(book_checkouts)
ebook_ranked = works_with_monthly_rank(ebook_checkouts)

In [4]:
print("=== Book_Ranked sample ===")
print(book_ranked.head())

print("\n=== EBook_ranked sample ===")
print(ebook_ranked.head())

=== Book_Ranked sample ===
                                            work_key  total_checkouts  \
0  TODAY WILL BE DIFFERENT : A NOVEL || SEMPLE, M...              898   
1  HILLBILLY ELEGY : A MEMOIR OF A FAMILY AND CUL...              596   
2  THE UNDERGROUND RAILROAD : A NOVEL || WHITEHEA...              542   
3              MOONGLOW : A NOVEL || CHABON, MICHAEL              442   
4  THE WRONG SIDE OF GOODBYE : A NOVEL || CONNELL...              400   

              genre                                              title  \
0             Humor                  TODAY WILL BE DIFFERENT : A NOVEL   
1  Biography/Memoir  HILLBILLY ELEGY : A MEMOIR OF A FAMILY AND CUL...   
2        Historical                 THE UNDERGROUND RAILROAD : A NOVEL   
3           Fiction                                 MOONGLOW : A NOVEL   
4  Mystery/Thriller                THE WRONG SIDE OF GOODBYE : A NOVEL   

             creator month_date  month_checkout_rank  checkoutyear  \
0      SEMPLE, MARI

### Develop Work Matching Logic

In [5]:
#method 1: match using exact work_key match - ebook and book work_key have to be identical
 
# Filter book_ranked to top 5 per month
book_top5_ranked = book_ranked[book_ranked['month_checkout_rank'] <= 5][
    ['work_key', 'title', 'creator', 'checkoutyear', 'checkoutmonth', 'genre', 'total_checkouts', 'month_checkout_rank']
].rename(columns={
    'total_checkouts':    'book_total_checkouts',
    'month_checkout_rank': 'book_rank'
})

# Join to ebook_ranked on work_key + month
crossed = book_top5_ranked.merge(
    ebook_ranked[['work_key', 'title', 'creator', 'checkoutyear', 'checkoutmonth', 'total_checkouts', 'month_checkout_rank']].rename(columns={
        'total_checkouts':    'ebook_total_checkouts',
        'month_checkout_rank': 'ebook_rank'
    }),
    on=['work_key', 'checkoutyear', 'checkoutmonth'],
    how='left'
)

# Flag ebook rank tiers
crossed['in_ebook_top5']  = crossed['ebook_rank'] <= 5
crossed['in_ebook_top10'] = crossed['ebook_rank'] <= 10
crossed['in_ebook_top20'] = crossed['ebook_rank'] <= 20
crossed['in_ebook_top50'] = crossed['ebook_rank'] <= 50

# Summary — how often do top 5 books appear in each ebook tier?
print("=== How often do top 5 physical books appear in ebook rankings? ===")
print(f"In ebook top 5:  {crossed['in_ebook_top5'].sum()} months")
print(f"In ebook top 10: {crossed['in_ebook_top10'].sum()} months")
print(f"In ebook top 20: {crossed['in_ebook_top20'].sum()} months")
print(f"In ebook top 50: {crossed['in_ebook_top50'].sum()} months")
print(f"Not in ebook rankings at all: {crossed['ebook_rank'].isna().sum()} months")

# View the actual matches
print("\n=== Works that appear in both top 5 books AND top 50 ebooks ===")
print(crossed[crossed['in_ebook_top50']].sort_values(['checkoutyear', 'checkoutmonth', 'book_rank'])
      .to_string(index=False))


=== How often do top 5 physical books appear in ebook rankings? ===
In ebook top 5:  0 months
In ebook top 10: 0 months
In ebook top 20: 0 months
In ebook top 50: 0 months
Not in ebook rankings at all: 540 months

=== Works that appear in both top 5 books AND top 50 ebooks ===
Empty DataFrame
Columns: [work_key, title_x, creator_x, checkoutyear, checkoutmonth, genre, book_total_checkouts, book_rank, title_y, creator_y, ebook_total_checkouts, ebook_rank, in_ebook_top5, in_ebook_top10, in_ebook_top20, in_ebook_top50]
Index: []


In [6]:
## method 2: fuzzy match on work_key with .8+ score
from rapidfuzz import process, fuzz

def fuzzy_match_work_key(book_key, ebook_keys, threshold=80):
    """Find best matching ebook work_key for a given book work_key"""
    match = process.extractOne(book_key, ebook_keys, 
                                scorer=fuzz.token_sort_ratio,
                                score_cutoff=threshold)
    if match:
        return match[0], match[1]  # matched key, score
    return None, None

# Get unique ebook work_keys per month for faster lookup
def fuzzy_cross_rank(book_df, ebook_df, book_rank_cutoff=5, threshold=80):
    results = []
    
    for (year, month), book_group in book_df[book_df['month_checkout_rank'] <= book_rank_cutoff].groupby(['checkoutyear', 'checkoutmonth']):
        ebook_month = ebook_df[(ebook_df['checkoutyear'] == year) & 
                                (ebook_df['checkoutmonth'] == month)]
        ebook_keys = ebook_month['work_key'].tolist()
        
        for _, book_row in book_group.iterrows():
            matched_key, score = fuzzy_match_work_key(book_row['work_key'], ebook_keys, threshold)
            
            if matched_key:
                ebook_row = ebook_month[ebook_month['work_key'] == matched_key].iloc[0]
                results.append({
                    'checkoutyear':        year,
                    'checkoutmonth':       month,
                    'book_work_key':       book_row['work_key'],
                    'ebook_work_key':      matched_key,
                    'match_score':         score,
                    'genre':               book_row['genre'],
                    'book_rank':           book_row['month_checkout_rank'],
                    'book_total_checkouts': book_row['total_checkouts'],
                    'ebook_rank':          ebook_row['month_checkout_rank'],
                    'ebook_total_checkouts': ebook_row['total_checkouts'],
                })
            else:
                results.append({
                    'checkoutyear':        year,
                    'checkoutmonth':       month,
                    'book_work_key':       book_row['work_key'],
                    'ebook_work_key':      None,
                    'match_score':         None,
                    'genre':               book_row['genre'],
                    'book_rank':           book_row['month_checkout_rank'],
                    'book_total_checkouts': book_row['total_checkouts'],
                    'ebook_rank':          None,
                    'ebook_total_checkouts': None,
                })
    
    return pd.DataFrame(results)

fuzzy_crossed = fuzzy_cross_rank(book_ranked, ebook_ranked, book_rank_cutoff=5, threshold=80)

print("=== Sample Matched Results ===")
print(fuzzy_crossed.sort_values(['checkoutyear', 'checkoutmonth', 'book_rank']).head().to_string(index=False))

total = len(fuzzy_crossed)
matched = fuzzy_crossed['ebook_rank'].notna().sum()

print("=== Method 2: How often do top 5 physical books appear in ebook rankings? ===")
print(f"Total book entries:    {total:,}")
print(f"Matched in ebooks:     {matched:,} ({matched/total*100:.1f}%)")
print(f"Not matched in ebooks: {total-matched:,} ({(total-matched)/total*100:.1f}%)")
print()

for tier in [5, 10, 20, 50]:
    count = (fuzzy_crossed['ebook_rank'] <= tier).sum()
    print(f"In ebook top {tier:>2}: {count:>5,} ({count/total*100:.1f}%)")



=== Sample Matched Results ===
 checkoutyear  checkoutmonth                                                                book_work_key                                                             ebook_work_key  match_score            genre  book_rank  book_total_checkouts  ebook_rank  ebook_total_checkouts
         2017              1                           TODAY WILL BE DIFFERENT : A NOVEL || SEMPLE, MARIA                                    TODAY WILL BE DIFFERENT || MARIA SEMPLE    87.640449            Humor          1                   898         3.0                  414.0
         2017              1 HILLBILLY ELEGY : A MEMOIR OF A FAMILY AND CULTURE IN CRISIS || VANCE, J. D. HILLBILLY ELEGY: A MEMOIR OF A FAMILY AND CULTURE IN CRISIS || J. D. VANCE    97.333333 Biography/Memoir          2                   596        24.0                  229.0
         2017              1                      THE UNDERGROUND RAILROAD : A NOVEL || WHITEHEAD, COLSON  THE UNDERGROUND RAILROAD 

In [7]:
#are there any noise words commonly found within the unmatched works?

from collections import Counter
import re

# Get unmatched book work keys
unmatched_keys = fuzzy_crossed[fuzzy_crossed['ebook_work_key'].isna()]['book_work_key'].tolist()

# Extract individual words and short phrases
word_counts = Counter()
bigram_counts = Counter()

for key in unmatched_keys:
    words = re.sub(r'[^a-z0-9\s]', ' ', key.lower()).split()
    # Single words
    word_counts.update(words)
    # Bigrams (2-word phrases)
    bigrams = [f"{words[i]} {words[i+1]}" for i in range(len(words)-1)]
    bigram_counts.update(bigrams)

print("=== Most common single words in unmatched titles ===")
for word, count in word_counts.most_common(20):
    print(f"{count:>5} | {word}")

print("\n=== Most common 2-word phrases in unmatched titles ===")
for bigram, count in bigram_counts.most_common(20):
    print(f"{count:>5} | {bigram}")

=== Most common single words in unmatched titles ===
   35 | the
   27 | a
   21 | novel
   12 | of
    8 | lee
    8 | jane
    7 | child
    7 | elizabeth
    6 | louise
    5 | connelly
    5 | michael
    5 | penny
    5 | and
    5 | ann
    4 | harper
    4 | big
    4 | tom
    4 | lake
    4 | patchett
    3 | wrong

=== Most common 2-word phrases in unmatched titles ===
   19 | a novel
    7 | child lee
    5 | connelly michael
    5 | penny louise
    5 | of the
    4 | harper jane
    4 | jane jane
    4 | jane elizabeth
    4 | tom lake
    4 | lake a
    4 | novel patchett
    4 | patchett ann
    3 | the wrong
    3 | wrong side
    3 | side of
    3 | of goodbye
    3 | goodbye a
    3 | novel connelly
    3 | less a
    3 | novel greer


In [8]:
sample_work = book_ranked['work_key'].iloc[0]
sample_work_cleaned = sample_work.lower().replace('novel', '').upper()

print(sample_work_cleaned)

print('Uncleaned Matches:')
print(process.extract(sample_work, ebook_ranked['work_key'].unique().tolist()) )

print('Cleaned Matches:')
print(process.extract(sample_work_cleaned, ebook_ranked['work_key'].unique().tolist()) )


TODAY WILL BE DIFFERENT : A  || SEMPLE, MARIA
Uncleaned Matches:
[('TODAY WILL BE DIFFERENT || MARIA SEMPLE', 85.6338028169014, 2), ('COMMONWEALTH || ANN PATCHETT', 85.5, 3), ('THE MARTIAN: A NOVEL || ANDY WEIR', 85.5, 5), ('THE LIFE-CHANGING MAGIC OF TIDYING UP: THE JAPANESE ART OF DECLUTTERING AND ORGANIZING || MARIE KONDO', 85.5, 6), ('ME BEFORE YOU || JOJO MOYES', 85.5, 7)]
Cleaned Matches:
[('TODAY WILL BE DIFFERENT || MARIA SEMPLE', 89.27710843373494, 2), ("THE UNDERGROUND RAILROAD (OPRAH'S BOOK CLUB): A NOVEL || COLSON WHITEHEAD", 85.5, 1), ('COMMONWEALTH || ANN PATCHETT', 85.5, 3), ('THE LIFE-CHANGING MAGIC OF TIDYING UP: THE JAPANESE ART OF DECLUTTERING AND ORGANIZING || MARIE KONDO', 85.5, 6), ('ME BEFORE YOU || JOJO MOYES', 85.5, 7)]


In [9]:
#method 3: fuzzy match on title score .8+, use author fuzzy match as tiebreaker if multiple matches

def fuzzy_cross_rank_title_author(book_df, ebook_df, book_rank_cutoff=5, title_threshold=80, author_threshold=80):
    results = []
    
    for (year, month), book_group in book_df[book_df['month_checkout_rank'] <= book_rank_cutoff].groupby(['checkoutyear', 'checkoutmonth']):
        ebook_month = ebook_df[(ebook_df['checkoutyear'] == year) & 
                                (ebook_df['checkoutmonth'] == month)].drop_duplicates('title')
        
        ebook_titles  = ebook_month['title'].tolist()
        ebook_authors = ebook_month['creator'].tolist()

        for _, book_row in book_group.iterrows():
            # Step 1 — find top title candidates above threshold
            title_candidates = process.extract(book_row['title'], ebook_titles,
                                               scorer=fuzz.token_sort_ratio, limit=5)
            
            # Filter to candidates that pass title threshold
            passing = [(title, score, idx) for title, score, idx in title_candidates
                       if score >= title_threshold]
            
            best_match      = None
            best_title_score  = None
            best_author_score = None

            if len(passing) == 0:
                # No title matches
                best_match = None

            elif len(passing) == 1:
                # Only one match — accept it without checking author
                _, best_title_score, idx = passing[0]
                best_match        = ebook_month.iloc[idx]
                best_author_score = None

            else:
                # Multiple title matches — use author as tiebreaker
                best_combined_score = 0
                for _, title_score, idx in passing:
                    ebook_author = ebook_authors[idx]
                    author_score = fuzz.token_sort_ratio(book_row['creator'], ebook_author)
                    combined_score = (title_score * 0.7) + (author_score * 0.3)
                    
                    if combined_score > best_combined_score:
                        best_combined_score = combined_score
                        best_match        = ebook_month.iloc[idx]
                        best_title_score  = title_score
                        best_author_score = author_score

            if best_match is not None:
                results.append({
                    'checkoutyear':          year,
                    'checkoutmonth':         month,
                    'book_title':            book_row['title'],
                    'ebook_title':           best_match['title'],
                    'book_author':           book_row['creator'],
                    'ebook_author':          best_match['creator'],
                    'title_score':           best_title_score,
                    'author_score':          best_author_score,
                    'genre':                 book_row['genre'],
                    'book_rank':             book_row['month_checkout_rank'],
                    'book_total_checkouts':  book_row['total_checkouts'],
                    'ebook_rank':            best_match['month_checkout_rank'],
                    'ebook_total_checkouts': best_match['total_checkouts'],
                })
            else:
                results.append({
                    'checkoutyear':          year,
                    'checkoutmonth':         month,
                    'book_title':            book_row['title'],
                    'ebook_title':           None,
                    'book_author':           book_row['creator'],
                    'ebook_author':          None,
                    'title_score':           None,
                    'author_score':          None,
                    'genre':                 book_row['genre'],
                    'book_rank':             book_row['month_checkout_rank'],
                    'book_total_checkouts':  book_row['total_checkouts'],
                    'ebook_rank':            None,
                    'ebook_total_checkouts': None,
                })
    
    return pd.DataFrame(results)

fuzzy_crossed_v2 = fuzzy_cross_rank_title_author(book_ranked, ebook_ranked, book_rank_cutoff=5)
print(fuzzy_crossed_v2.sort_values(['checkoutyear', 'checkoutmonth', 'book_rank']).head().to_string(index=False))

total = len(fuzzy_crossed_v2)
matched = fuzzy_crossed_v2['ebook_rank'].notna().sum()

print("\n=== Method 3: How often do top 5 physical books appear in ebook rankings with title + author tiebreak matching? ===")
print(f"Total book entries:    {total:,}")
print(f"Matched in ebooks:     {matched:,} ({matched/total*100:.1f}%)")
print(f"Not matched in ebooks: {total-matched:,} ({(total-matched)/total*100:.1f}%)")
print()

for tier in [5, 10, 20, 50]:
    count = (fuzzy_crossed_v2['ebook_rank'] <= tier).sum()
    print(f"In ebook top {tier:>2}: {count:>5,} ({count/total*100:.1f}%)")

 checkoutyear  checkoutmonth                                                   book_title                                                 ebook_title       book_author   ebook_author  title_score  author_score            genre  book_rank  book_total_checkouts  ebook_rank  ebook_total_checkouts
         2017              1                            TODAY WILL BE DIFFERENT : A NOVEL                                     TODAY WILL BE DIFFERENT     SEMPLE, MARIA   MARIA SEMPLE    82.142857           NaN            Humor          1                   898         3.0                  414.0
         2017              1 HILLBILLY ELEGY : A MEMOIR OF A FAMILY AND CULTURE IN CRISIS HILLBILLY ELEGY: A MEMOIR OF A FAMILY AND CULTURE IN CRISIS      VANCE, J. D.    J. D. VANCE    97.478992           NaN Biography/Memoir          2                   596        24.0                  229.0
         2017              1                           THE UNDERGROUND RAILROAD : A NOVEL                          

In [10]:
##step by step debug of unmatched titles

from rapidfuzz import process, fuzz

# Pick a specific month to debug
year, month = 2017, 1

# Get unmatched book titles for that month from fuzzy_crossed_v2
unmatched_titles = fuzzy_crossed_v2[
    (fuzzy_crossed_v2['checkoutyear'] == year) &
    (fuzzy_crossed_v2['checkoutmonth'] == month) &
    (fuzzy_crossed_v2['ebook_rank'].isna())
]['book_title'].tolist()

# Look up those titles in book_ranked
book_sample = book_ranked[
    (book_ranked['checkoutyear'] == year) &
    (book_ranked['checkoutmonth'] == month) &
    (book_ranked['title'].isin(unmatched_titles))
]

# All ebooks for that month — no filtering needed
ebook_sample = ebook_ranked[
    (ebook_ranked['checkoutyear'] == year) &
    (ebook_ranked['checkoutmonth'] == month)
].drop_duplicates('title')

print("=== Top 5 book titles ===")
print(book_sample[['title', 'creator']].to_string(index=False))

print("\n=== Top 10 ebook titles ===")
print(ebook_sample[['title', 'creator']].head(10).to_string(index=False))

print("\n=== Fuzzy match results on work key for each book title ===")
for _, row in book_sample.iterrows():
    print(f"\nBook: {row['work_key']}")
    matches = process.extract(row['work_key'], ebook_sample['work_key'].tolist(),
                               scorer=fuzz.token_sort_ratio, limit=5)
    for matched_work_key, score, idx in matches:
        ebook_work_key = ebook_sample['work_key'].tolist()[idx]
        print(f"  Score: {score:>5.1f} | {matched_work_key} ")

print("\n=== Fuzzy match results on title + author (tiebreak) for each book title ===")
for _, row in book_sample.iterrows():
    print(f"\nBook: {row['title']} | {row['creator']}")
    matches = process.extract(row['title'], ebook_sample['title'].tolist(),
                               scorer=fuzz.token_sort_ratio, limit=5)
    for matched_title, score, idx in matches:
        ebook_author = ebook_sample['creator'].tolist()[idx]
        author_score = fuzz.token_sort_ratio(row['creator'], ebook_author)
        print(f"  Score: {score:>5.1f} | Author: {author_score:>5.1f} | {matched_title} | {ebook_author}")

=== Top 5 book titles ===
                              title           creator
 THE UNDERGROUND RAILROAD : A NOVEL WHITEHEAD, COLSON
THE WRONG SIDE OF GOODBYE : A NOVEL CONNELLY, MICHAEL

=== Top 10 ebook titles ===
                                                                                 title          creator
                                   THE GOLDFINCH: A NOVEL (PULITZER PRIZE FOR FICTION)      DONNA TARTT
                                 THE UNDERGROUND RAILROAD (OPRAH'S BOOK CLUB): A NOVEL COLSON WHITEHEAD
                                                               TODAY WILL BE DIFFERENT     MARIA SEMPLE
                                                                          COMMONWEALTH     ANN PATCHETT
                                                              BETWEEN THE WORLD AND ME TA-NEHISI COATES
                                                                  THE MARTIAN: A NOVEL        ANDY WEIR
THE LIFE-CHANGING MAGIC OF TIDYING UP: THE JAPANESE ART

In [11]:
#method 4 - Fuzzy Match on title, break match ties on author
from rapidfuzz import process, fuzz

def fuzzy_cross_rank_title_author(book_df, ebook_df, book_rank_cutoff=5, tie_threshold=2):
    """
    Match on title score — take the best match.
    Use author as tiebreaker if top scores are within tie_threshold points of each other.
    """
    results = []
    
    for (year, month), book_group in book_df[book_df['month_checkout_rank'] <= book_rank_cutoff].groupby(['checkoutyear', 'checkoutmonth']):
        ebook_month = ebook_df[(ebook_df['checkoutyear'] == year) & 
                                (ebook_df['checkoutmonth'] == month)].drop_duplicates('title')
        
        ebook_titles  = ebook_month['title'].tolist()
        ebook_authors = ebook_month['creator'].tolist()

        for _, book_row in book_group.iterrows():
            title_candidates = process.extract(book_row['title'], ebook_titles,
                                               scorer=fuzz.token_sort_ratio, limit=5)
            
            if not title_candidates:
                best_match        = None
                best_title_score  = None
                best_author_score = None
            else:
                top_score = title_candidates[0][1]
                
                # Find all candidates within tie_threshold of the top score
                tied = [(title, score, idx) for title, score, idx in title_candidates
                        if top_score - score <= tie_threshold]
                
                if len(tied) == 1:
                    # Clear winner — take it without checking author
                    _, best_title_score, idx = tied[0]
                    best_match        = ebook_month.iloc[idx]
                    best_author_score = None
                else:
                    # Tied — use author as tiebreaker
                    best_combined_score = 0
                    best_match          = None
                    best_title_score    = None
                    best_author_score   = None
                    for _, title_score, idx in tied:
                        author_score   = fuzz.token_sort_ratio(book_row['creator'], ebook_authors[idx])
                        combined_score = (title_score * 0.7) + (author_score * 0.3)
                        if combined_score > best_combined_score:
                            best_combined_score = combined_score
                            best_match          = ebook_month.iloc[idx]
                            best_title_score    = title_score
                            best_author_score   = author_score

            if best_match is not None:
                results.append({
                    'checkoutyear':          year,
                    'checkoutmonth':         month,
                    'book_title':            book_row['title'],
                    'ebook_title':           best_match['title'],
                    'book_author':           book_row['creator'],
                    'ebook_author':          best_match['creator'],
                    'title_score':           best_title_score,
                    'author_score':          best_author_score,
                    'genre':                 book_row['genre'],
                    'book_rank':             book_row['month_checkout_rank'],
                    'book_total_checkouts':  book_row['total_checkouts'],
                    'ebook_rank':            best_match['month_checkout_rank'],
                    'ebook_total_checkouts': best_match['total_checkouts'],
                })
            else:
                results.append({
                    'checkoutyear':          year,
                    'checkoutmonth':         month,
                    'book_title':            book_row['title'],
                    'ebook_title':           None,
                    'book_author':           book_row['creator'],
                    'ebook_author':          None,
                    'title_score':           None,
                    'author_score':          None,
                    'genre':                 book_row['genre'],
                    'book_rank':             book_row['month_checkout_rank'],
                    'book_total_checkouts':  book_row['total_checkouts'],
                    'ebook_rank':            None,
                    'ebook_total_checkouts': None,
                })
    
    return pd.DataFrame(results)

fuzzy_crossed_v3 = fuzzy_cross_rank_title_author(book_ranked, ebook_ranked, book_rank_cutoff=5)

#this should by default match every book - confirm
total = len(fuzzy_crossed_v3)
matched = fuzzy_crossed_v3['ebook_rank'].notna().sum()

print("\n=== Method 4: How often do top 5 physical books appear in ebook rankings with closest title + author tiebreak matching? ===")
print(f"Total book entries:    {total:,}")
print(f"Matched in ebooks:     {matched:,} ({matched/total*100:.1f}%)")
print(f"Not matched in ebooks: {total-matched:,} ({(total-matched)/total*100:.1f}%)")
print()

for tier in [5, 10, 20, 50]:
    count = (fuzzy_crossed_v3['ebook_rank'] <= tier).sum()
    print(f"In ebook top {tier:>2}: {count:>5,} ({count/total*100:.1f}%)")


=== Method 4: How often do top 5 physical books appear in ebook rankings with closest title + author tiebreak matching? ===
Total book entries:    540
Matched in ebooks:     540 (100.0%)
Not matched in ebooks: 0 (0.0%)

In ebook top  5:    57 (10.6%)
In ebook top 10:    81 (15.0%)
In ebook top 20:   135 (25.0%)
In ebook top 50:   204 (37.8%)


In [12]:
# validate
# spot check the 10 lowest scoring matches
worst_matches = fuzzy_crossed_v3.nsmallest(10, 'title_score')[['book_title', 'ebook_title', 'book_author', 'ebook_author', 'title_score', 'author_score']]

print("=== 10 Lowest Title Scoring Matches ===")
print(worst_matches.to_string(index=False))

# For each worst match, show all 5 candidates that were considered
print("\n=== All candidates considered for each match ===")
for _, row in worst_matches.iterrows():
    print(f"\nBook:  {row['book_title']} | {row['book_author']}")
    print(f"Chose: {row['ebook_title']} | {row['ebook_author']} (score: {row['title_score']})")
    
    # Get the ebook titles for that month
    year  = fuzzy_crossed_v3[fuzzy_crossed_v3['book_title'] == row['book_title']]['checkoutyear'].iloc[0]
    month = fuzzy_crossed_v3[fuzzy_crossed_v3['book_title'] == row['book_title']]['checkoutmonth'].iloc[0]
    
    ebook_month  = ebook_ranked[(ebook_ranked['checkoutyear'] == year) &
                                 (ebook_ranked['checkoutmonth'] == month)].drop_duplicates('title')
    ebook_titles = ebook_month['title'].tolist()
    
    candidates = process.extract(row['book_title'], ebook_titles, scorer=fuzz.token_sort_ratio, limit=5)
    print("  All candidates:")
    for matched_title, score, idx in candidates:
        author_score = fuzz.token_sort_ratio(row['book_author'], ebook_month['creator'].tolist()[idx])
        print(f"  {score:>5.1f} title | {author_score:>5.1f} author | {matched_title}")

=== 10 Lowest Title Scoring Matches ===
                                                                                                                                                  book_title                                                                                                                       ebook_title            book_author      ebook_author  title_score  author_score
                                                                              SELOC HONDA OUTBOARDS : 1978-99 REPAIR MANUAL, 2-130 HORSEPOWER, 1-4 CYCLINDER                                                                                  STITCHES: A HANDBOOK ON MEANING, HOPE AND REPAIR                   None       ANNE LAMOTT    46.031746     13.333333
                                                                                    SEMBRANDO HISTORIAS : PURA BELPRÉ : BIBLIOTECARIA Y NARRADORA DE CUENTOS                                                                                              

In [13]:
#the lowest 10 match scores are wrong.  Validate this across the entire dataset

def validate_match(book_title, ebook_title, book_author, ebook_author, 
                   author_threshold=70, min_substring_len=3):
    """
    Validate a fuzzy match by:
    1. Check if authors match above threshold
    2. If yes, check if any meaningful word from book title appears in ebook title
    """
    # Step 1 — author match
    author_score = fuzz.token_sort_ratio(str(book_author), str(ebook_author))
    if author_score < author_threshold:
        return False, author_score, None
    
    # Step 2 — substring check: any word from book title in ebook title?
    book_words = set(re.sub(r'[^a-z0-9\s]', ' ', str(book_title).lower()).split())
    ebook_title_lower = str(ebook_title).lower()
    
    # Filter out noise words
    noise = {'a', 'an', 'the', 'of', 'in', 'and', 'or', 'to', 'by', 'for', 
             'novel', 'memoir', 'story', 'book', 'vol', 'volume'}
    meaningful_words = [w for w in book_words if w not in noise and len(w) >= min_substring_len]
    
    matched_words = [w for w in meaningful_words if w in ebook_title_lower]
    
    if matched_words:
        return True, author_score, matched_words
    return False, author_score, None

# Apply validation to fuzzy_crossed_v3
fuzzy_crossed_v3['valid_match'] = False
fuzzy_crossed_v3['validated_author_score'] = None
fuzzy_crossed_v3['matched_words'] = None

for idx, row in fuzzy_crossed_v3[fuzzy_crossed_v3['ebook_title'].notna()].iterrows():
    is_valid, author_score, matched_words = validate_match(
        row['book_title'], row['ebook_title'],
        row['book_author'], row['ebook_author']
    )
    fuzzy_crossed_v3.at[idx, 'valid_match']           = is_valid
    fuzzy_crossed_v3.at[idx, 'validated_author_score'] = author_score
    fuzzy_crossed_v3.at[idx, 'matched_words']          = str(matched_words) if matched_words else None

# Summary
total    = len(fuzzy_crossed_v3)
matched  = fuzzy_crossed_v3['ebook_title'].notna().sum()
valid    = fuzzy_crossed_v3['valid_match'].sum()
invalid  = matched - valid

print("=== Match Validation Summary ===")
print(f"Total book entries:       {total:,}")
print(f"Fuzzy matched:            {matched:,} ({matched/total*100:.1f}%)")
print(f"Validated (author+word):  {valid:,} ({valid/total*100:.1f}%)")
print(f"Invalid matches:          {invalid:,} ({invalid/total*100:.1f}%)")
print(f"No match found:           {total-matched:,} ({(total-matched)/total*100:.1f}%)")

print("\n=== Sample valid matches ===")
print(fuzzy_crossed_v3[fuzzy_crossed_v3['valid_match']][
    ['book_title', 'ebook_title', 'book_author', 'ebook_author', 'validated_author_score', 'matched_words']
].head(10).to_string(index=False))

print("\n=== Sample invalid matches ===")
print(fuzzy_crossed_v3[(fuzzy_crossed_v3['ebook_title'].notna()) & 
                        (~fuzzy_crossed_v3['valid_match'])][
    ['book_title', 'ebook_title', 'book_author', 'ebook_author', 'validated_author_score', 'matched_words']
].head(10).to_string(index=False))

=== Match Validation Summary ===
Total book entries:       540
Fuzzy matched:            540 (100.0%)
Validated (author+word):  369 (68.3%)
Invalid matches:          171 (31.7%)
No match found:           0 (0.0%)

=== Sample valid matches ===
                                                  book_title                                                 ebook_title       book_author     ebook_author validated_author_score                                         matched_words
                           TODAY WILL BE DIFFERENT : A NOVEL                                     TODAY WILL BE DIFFERENT     SEMPLE, MARIA     MARIA SEMPLE                   96.0                        ['today', 'different', 'will']
HILLBILLY ELEGY : A MEMOIR OF A FAMILY AND CULTURE IN CRISIS HILLBILLY ELEGY: A MEMOIR OF A FAMILY AND CULTURE IN CRISIS      VANCE, J. D.      J. D. VANCE              95.652174 ['crisis', 'elegy', 'family', 'hillbilly', 'culture']
                          THE UNDERGROUND RAILROAD : A NOV

In [14]:
# For each matched book, check if the correct author exists in ebook_ranked for that month
results = []

for _, row in fuzzy_crossed_v3[fuzzy_crossed_v3['ebook_title'].notna()].head(50).iterrows():
    year  = row['checkoutyear']
    month = row['checkoutmonth']
    
    # Get all ebooks for that month
    ebook_month = ebook_ranked[(ebook_ranked['checkoutyear'] == year) &
                                (ebook_ranked['checkoutmonth'] == month)]
    
    # Check if correct author exists in ebook catalog that month
    author_score_max = ebook_month['creator'].apply(
        lambda a: fuzz.token_sort_ratio(row['book_author'], str(a))
    ).max()
    
    # Check if correct title exists in ebook catalog that month
    title_score_max = ebook_month['title'].apply(
        lambda t: fuzz.token_sort_ratio(row['book_title'], str(t))
    ).max()
    
    results.append({
        'book_title':         row['book_title'],
        'matched_ebook':      row['ebook_title'],
        'book_author':        row['book_author'],
        'matched_author':     row['ebook_author'],
        'title_score':        row['title_score'],
        'matched_author_score': fuzz.token_sort_ratio(row['book_author'], row['ebook_author']),
        'best_author_in_catalog': author_score_max,
        'best_title_in_catalog':  title_score_max,
    })

df_check = pd.DataFrame(results)

print("=== Title match returned wrong author? ===")
# Cases where the matched author is bad but a better author exists in catalog
wrong_author = df_check[
    (df_check['matched_author_score'] < 70) &
    (df_check['best_author_in_catalog'] >= 70)
]
print(f"Matched to wrong author but correct author exists: {len(wrong_author):,}")
print(wrong_author[['book_title', 'matched_ebook', 'book_author', 'matched_author',
                     'matched_author_score', 'best_author_in_catalog', 'best_title_in_catalog']
].to_string(index=False))

=== Title match returned wrong author? ===
Matched to wrong author but correct author exists: 13
                         book_title                     matched_ebook       book_author   matched_author  matched_author_score  best_author_in_catalog  best_title_in_catalog
THE WRONG SIDE OF GOODBYE : A NOVEL       THE ONE GOOD THING: A NOVEL CONNELLY, MICHAEL KEVIN ALAN MILNE             36.363636               96.969697              70.967742
THE WRONG SIDE OF GOODBYE : A NOVEL THE LOVE OF A GOOD WOMAN: STORIES CONNELLY, MICHAEL      ALICE MUNRO             28.571429               96.969697              73.529412
THE WRONG SIDE OF GOODBYE : A NOVEL THE LOVE OF A GOOD WOMAN: STORIES CONNELLY, MICHAEL      ALICE MUNRO             28.571429               96.969697              73.529412
                       NIGHT SCHOOL                        NIGHT SONG        CHILD, LEE  BEVERLY JENKINS             24.000000               94.736842              72.727273
                     INTO THE WAT

In [15]:
## Method 5 Fuzzy Match on title, break match ties on author, ADD validation criteria: 
### Check if authors match above threshold
### check if 3+ meaningful words from book title appears in ebook title

def validate_match(book_title, ebook_title, book_author, ebook_author,
                   author_threshold=70, min_substring_len=3):
    author_score = fuzz.token_sort_ratio(str(book_author), str(ebook_author))
    if author_score < author_threshold:
        return False, author_score, None
    
    book_words        = set(re.sub(r'[^a-z0-9\s]', ' ', str(book_title).lower()).split())
    ebook_title_lower = str(ebook_title).lower()
    noise             = {'a', 'an', 'the', 'of', 'in', 'and', 'or', 'to', 'by', 'for',
                         'novel', 'memoir', 'story', 'book', 'vol', 'volume'}
    meaningful_words  = [w for w in book_words if w not in noise and len(w) >= min_substring_len]
    matched_words     = [w for w in meaningful_words if w in ebook_title_lower]
    
    if matched_words:
        return True, author_score, matched_words
    return False, author_score, None


def fuzzy_cross_rank_title_author(book_df, ebook_df, book_rank_cutoff=5, tie_threshold=2,
                                   author_threshold=70, min_substring_len=3):
    results = []
    
    for (year, month), book_group in book_df[book_df['month_checkout_rank'] <= book_rank_cutoff].groupby(['checkoutyear', 'checkoutmonth']):
        ebook_month  = ebook_df[(ebook_df['checkoutyear'] == year) &
                                 (ebook_df['checkoutmonth'] == month)].drop_duplicates('title')
        ebook_titles  = ebook_month['title'].tolist()
        ebook_authors = ebook_month['creator'].tolist()

        for _, book_row in book_group.iterrows():
            title_candidates = process.extract(book_row['title'], ebook_titles,
                                               scorer=fuzz.token_sort_ratio, limit=5)
            
            if not title_candidates:
                best_match = best_title_score = best_author_score = best_matched_words = None
            else:
                top_score = title_candidates[0][1]
                tied      = [(title, score, idx) for title, score, idx in title_candidates
                             if top_score - score <= tie_threshold]
                
                best_match = best_title_score = best_author_score = best_matched_words = None

                if len(tied) == 1:
                    _, best_title_score, idx = tied[0]
                    candidate = ebook_month.iloc[idx]
                    is_valid, best_author_score, best_matched_words = validate_match(
                        book_row['title'], candidate['title'],
                        book_row['creator'], candidate['creator'],
                        author_threshold, min_substring_len
                    )
                    if is_valid:
                        best_match = candidate
                else:
                    best_combined_score = 0
                    for _, title_score, idx in tied:
                        candidate = ebook_month.iloc[idx]
                        is_valid, author_score, matched_words = validate_match(
                            book_row['title'], candidate['title'],
                            book_row['creator'], candidate['creator'],
                            author_threshold, min_substring_len
                        )
                        if not is_valid:
                            continue
                        combined_score = (title_score * 0.7) + (author_score * 0.3)
                        if combined_score > best_combined_score:
                            best_combined_score = combined_score
                            best_match          = candidate
                            best_title_score    = title_score
                            best_author_score   = author_score
                            best_matched_words  = matched_words

            if best_match is not None:
                results.append({
                    'checkoutyear':          year,
                    'checkoutmonth':         month,
                    'book_title':            book_row['title'],
                    'ebook_title':           best_match['title'],
                    'book_author':           book_row['creator'],
                    'ebook_author':          best_match['creator'],
                    'title_score':           best_title_score,
                    'author_score':          best_author_score,
                    'matched_words':         str(best_matched_words),
                    'genre':                 book_row['genre'],
                    'book_rank':             book_row['month_checkout_rank'],
                    'book_total_checkouts':  book_row['total_checkouts'],
                    'ebook_rank':            best_match['month_checkout_rank'],
                    'ebook_total_checkouts': best_match['total_checkouts'],
                })
            else:
                results.append({
                    'checkoutyear':          year,
                    'checkoutmonth':         month,
                    'book_title':            book_row['title'],
                    'ebook_title':           None,
                    'book_author':           book_row['creator'],
                    'ebook_author':          None,
                    'title_score':           None,
                    'author_score':          None,
                    'matched_words':         None,
                    'genre':                 book_row['genre'],
                    'book_rank':             book_row['month_checkout_rank'],
                    'book_total_checkouts':  book_row['total_checkouts'],
                    'ebook_rank':            None,
                    'ebook_total_checkouts': None,
                })
    
    return pd.DataFrame(results)

fuzzy_crossed_v4 = fuzzy_cross_rank_title_author(book_ranked, ebook_ranked, book_rank_cutoff=5)

total = len(fuzzy_crossed_v4)
matched = fuzzy_crossed_v4['ebook_rank'].notna().sum()

print("=== How often do top 5 physical books appear in ebook rankings with title + author tiebreak matching? ===")
print(f"Total book entries:    {total:,}")
print(f"Matched in ebooks:     {matched:,} ({matched/total*100:.1f}%)")
print(f"Not matched in ebooks: {total-matched:,} ({(total-matched)/total*100:.1f}%)")
print()

for tier in [5, 10, 20, 50]:
    count = (fuzzy_crossed_v4['ebook_rank'] <= tier).sum()
    print(f"In ebook top {tier:>2}: {count:>5,} ({count/total*100:.1f}%)")

=== How often do top 5 physical books appear in ebook rankings with title + author tiebreak matching? ===
Total book entries:    540
Matched in ebooks:     369 (68.3%)
Not matched in ebooks: 171 (31.7%)

In ebook top  5:    56 (10.4%)
In ebook top 10:    80 (14.8%)
In ebook top 20:   134 (24.8%)
In ebook top 50:   201 (37.2%)


In [16]:
# check unmatched results

unmatched = fuzzy_crossed_v4[fuzzy_crossed_v4['ebook_rank'].isna()].drop_duplicates('book_title')

candidates_list = []

for _, row in unmatched.iterrows():
    year  = row['checkoutyear']
    month = row['checkoutmonth']
    
    ebook_month   = ebook_ranked[(ebook_ranked['checkoutyear'] == year) &
                                  (ebook_ranked['checkoutmonth'] == month)].drop_duplicates('title')
    ebook_titles  = ebook_month['title'].tolist()
    ebook_authors = ebook_month['creator'].tolist()
    
    candidates = process.extract(row['book_title'], ebook_titles,
                                  scorer=fuzz.token_sort_ratio, limit=5)
    
    # Label each candidate by rank (1 = closest)
    for rank, (matched_title, title_score, idx) in enumerate(candidates, start=1):
        ebook_author = ebook_authors[idx]
        author_score = fuzz.token_sort_ratio(row['book_author'], ebook_author)
        combined     = (title_score * 0.7) + (author_score * 0.3)
        _, _, matched_words = validate_match(
            row['book_title'], matched_title,
            row['book_author'], ebook_author
        )
        candidates_list.append({
            'book_title':     row['book_title'],
            'book_author':    row['book_author'],
            'ebook_title':    matched_title,
            'ebook_author':   ebook_author,
            'ebook_candidate_rank': rank,
            'title_score':    round(title_score, 1),
            'author_score':   round(author_score, 1),
            'combined_score': round(combined, 1),
            'matched_words':  matched_words
        })

candidates_df = pd.DataFrame(candidates_list)

# Get top 20 book-ebook pairs by combined score
top20_books = (candidates_df.sort_values('combined_score', ascending=False)
               .drop_duplicates('book_title')
               .head(20)['book_title'].tolist())

# Print top 20 books with all 5 ebook candidates
print("=== Top 20 Unmatched Books by Best Combined Score — All 5 Ebook Candidates ===\n")
for book_title in top20_books:
    book_candidates = candidates_df[candidates_df['book_title'] == book_title].sort_values('ebook_candidate_rank')
    best = book_candidates.iloc[0]
    print(f"Book:   {best['book_title']}")
    print(f"Author: {best['book_author']}")
    print(f"  {'Rank':<6} {'Title Score':>11} {'Author Score':>12} {'Combined':>9} {'Matched Words':<20} Ebook Title | Ebook Author")
    for _, c in book_candidates.iterrows():
        print(f"  {c['ebook_candidate_rank']:<6} {c['title_score']:>11} {c['author_score']:>12} {c['combined_score']:>9} {str(c['matched_words']):<20} {c['ebook_title']} | {c['ebook_author']}")
    print()

=== Top 20 Unmatched Books by Best Combined Score — All 5 Ebook Candidates ===

Book:   THE FIFTH RISK
Author: LEWIS, MICHAEL (MICHAEL M.)
  Rank   Title Score Author Score  Combined Matched Words        Ebook Title | Ebook Author
  1            100.0         65.0      89.5 None                 THE FIFTH RISK | MICHAEL LEWIS
  2             80.0         25.6      63.7 None                 THE KING'S FIFTH | SCOTT O'DELL
  3             72.0         26.3      58.3 None                 THE LIFTERS | DAVE EGGERS
  4             69.2         30.0      57.5 None                 THE FIGHTERS | C. J. CHIVERS
  5             69.2         20.5      54.6 None                 THE GRIFTERS | JIM THOMPSON

Book:   GOING INFINITE : THE RISE AND FALL OF A NEW TYCOON
Author: LEWIS, MICHAEL (MICHAEL M.)
  Rank   Title Score Author Score  Combined Matched Words        Ebook Title | Ebook Author
  1             97.0         65.0      87.4 None                 GOING INFINITE: THE RISE AND FALL OF A NEW TY

In [17]:
## Method 6 
### Return closest fuzzy match author
### Return closest fuzzy match title that matches 75%+ of meaningful words from the book title

def fuzzy_cross_rank_v2(book_df, ebook_df, book_rank_cutoff=5,
                         author_threshold=70, min_substring_len=3):
    """
    Match on author first, then title.
    Return closest title match that also shares at least 1 meaningful word.
    """
    results = []
    
    noise = {'a', 'an', 'the', 'of', 'in', 'and', 'or', 'to', 'by', 'for',
             'novel', 'memoir', 'story', 'book', 'vol', 'volume'}

    def clean_author(author):
        author = re.sub(r'\(.*?\)', '', str(author))
        author = re.sub(r',?\s*(JR|SR|II|III|IV)\.?', '', author, flags=re.IGNORECASE)
        author = re.sub(r',?\s*\d{4}-?(\d{4})?', '', author)
        return author.strip()

    def get_meaningful_words(title, author=None):
        words = set(re.sub(r'[^a-z0-9\s]', ' ', str(title).lower()).split())
        author_words = set(re.sub(r'[^a-z0-9\s]', ' ', str(author).lower()).split()) if author else set()
        return [w for w in words 
                if w not in noise 
                and len(w) >= min_substring_len
                and w not in author_words]

    for (year, month), book_group in book_df[book_df['month_checkout_rank'] <= book_rank_cutoff].groupby(['checkoutyear', 'checkoutmonth']):
        ebook_month   = ebook_df[(ebook_df['checkoutyear'] == year) &
                                  (ebook_df['checkoutmonth'] == month)].drop_duplicates('title')
        ebook_authors = ebook_month['creator'].tolist()

        for _, book_row in book_group.iterrows():
            book_author_clean = clean_author(book_row['creator'])

            # Step 1 — fuzzy match on author, filter above threshold
            author_candidates = process.extract(book_author_clean, ebook_authors,
                                                 scorer=fuzz.token_sort_ratio, limit=10)
            passing_authors = [(score, idx) for _, score, idx in author_candidates
                               if score >= author_threshold]

            if not passing_authors:
                best_match = best_title_score = best_author_score = best_matched_words = None
            else:
                # Step 2 — among passing authors, score each title using WRatio
                # WRatio handles short vs long title comparisons better than token_sort_ratio
                title_candidates = []
                book_words = get_meaningful_words(book_row['title'], book_row['creator'])
                min_words_required = max(1, len(book_words) * .75)  # match at least 75% of meaningful words

                for author_score, idx in passing_authors:
                    candidate     = ebook_month.iloc[idx]
                    title_score   = fuzz.WRatio(book_row['title'], candidate['title'])
                    ebook_lower   = candidate['title'].lower()
                    matched_words = [w for w in book_words if w in ebook_lower]

                    # Only consider candidates matching >= half # meaningful words in book title
                    if len(matched_words) >= min_words_required:
                        title_candidates.append((title_score, author_score, idx, matched_words))

                if not title_candidates:
                    best_match = best_title_score = best_author_score = best_matched_words = None
                else:
                    # Take the candidate with the highest title score
                    title_candidates.sort(key=lambda x: x[0], reverse=True)
                    best_title_score, best_author_score, idx, best_matched_words = title_candidates[0]
                    best_match = ebook_month.iloc[idx]

            if best_match is not None:
                results.append({
                    'checkoutyear':          year,
                    'checkoutmonth':         month,
                    'month_date':            book_row['month_date'],
                    'book_title':            book_row['title'],
                    'ebook_title':           best_match['title'],
                    'book_author':           book_row['creator'],
                    'ebook_author':          best_match['creator'],
                    'title_score':           best_title_score,
                    'author_score':          best_author_score,
                    'matched_words':         str(best_matched_words),
                    'genre':                 book_row['genre'],
                    'book_rank':             book_row['month_checkout_rank'],
                    'book_total_checkouts':  book_row['total_checkouts'],
                    'ebook_rank':            best_match['month_checkout_rank'],
                    'ebook_total_checkouts': best_match['total_checkouts'],
                })
            else:
                results.append({
                    'checkoutyear':          year,
                    'checkoutmonth':         month,
                    'month_date':            book_row['month_date'],
                    'book_title':            book_row['title'],
                    'ebook_title':           None,
                    'book_author':           book_row['creator'],
                    'ebook_author':          None,
                    'title_score':           None,
                    'author_score':          None,
                    'matched_words':         None,
                    'genre':                 book_row['genre'],
                    'book_rank':             book_row['month_checkout_rank'],
                    'book_total_checkouts':  book_row['total_checkouts'],
                    'ebook_rank':            None,
                    'ebook_total_checkouts': None,
                })

    return pd.DataFrame(results)

fuzzy_crossed_v6 = fuzzy_cross_rank_v2(book_ranked, ebook_ranked, book_rank_cutoff=5)

total   = len(fuzzy_crossed_v6)
matched = fuzzy_crossed_v6['ebook_rank'].notna().sum()

print("=== Method 6: Fuzzy author match → closest title with 1+ meaningful word ===")
print(f"Total book entries:    {total:,}")
print(f"Matched in ebooks:     {matched:,} ({matched/total*100:.1f}%)")
print(f"Not matched in ebooks: {total-matched:,} ({(total-matched)/total*100:.1f}%)")
print()
for tier in [5, 10, 20, 50]:
    count = (fuzzy_crossed_v6['ebook_rank'] <= tier).sum()
    print(f"In ebook top {tier:>2}: {count:>5,} ({count/total*100:.1f}%)")

=== Method 6: Fuzzy author match → closest title with 1+ meaningful word ===
Total book entries:    540
Matched in ebooks:     493 (91.3%)
Not matched in ebooks: 47 (8.7%)

In ebook top  5:    85 (15.7%)
In ebook top 10:   116 (21.5%)
In ebook top 20:   182 (33.7%)
In ebook top 50:   275 (50.9%)


In [18]:
##check lowest similarity score matches to confirm they're correct
matched = fuzzy_crossed_v6[fuzzy_crossed_v6['ebook_rank'].notna()].copy()
matched['num_matched_words'] = matched['matched_words'].apply(
    lambda x: len(eval(x)) if x is not None and x != 'None' else 0
)

# Weight: author most important, then title, then word count
matched['combined_score'] = (
    (matched['author_score'] * 0.5) +
    (matched['title_score']  * 0.4) +
    (matched['num_matched_words'] * 2)  # small boost per word
)

worst = (matched
         .drop_duplicates('book_title')
         .nsmallest(20, 'combined_score'))

print("=== 20 Lowest Combined Score Matches ===")
for _, row in worst.iterrows():
    print(f"{'─' * 80}")
    print(f"Book:          {row['book_title'][:75]}")
    print(f"Ebook:         {row['ebook_title'][:75]}")
    print(f"Book Author:   {row['book_author'][:50]}")
    print(f"Ebook Author:  {row['ebook_author'][:50]}")
    print(f"Title Score:   {row['title_score']:.1f}  |  Author Score: {row['author_score']:.1f}  |  Matched Words: {row['num_matched_words']}")
    print(f"Words Matched: {row['matched_words']}")
    print(f"Combined Score: {row['combined_score']}")
print(f"{'─' * 80}")

=== 20 Lowest Combined Score Matches ===
────────────────────────────────────────────────────────────────────────────────
Book:          HELLO BEAUTIFUL : A NOVEL
Ebook:         HELLO BEAUTIFUL: OPRAH'S BOOK CLUB
Book Author:   NAPOLITANO, ANN
Ebook Author:  ANN NAPOLITANO
Title Score:   71.2  |  Author Score: 96.6  |  Matched Words: 2
Words Matched: ['beautiful', 'hello']
Combined Score: 80.75043834015196
────────────────────────────────────────────────────────────────────────────────
Book:          THE LIFE IMPOSSIBLE
Ebook:         THE LIFE IMPOSSIBLE: A NOVEL
Book Author:   HAIG, MATT
Ebook Author:  MATT HAIG
Title Score:   80.9  |  Author Score: 94.7  |  Matched Words: 2
Words Matched: ['life', 'impossible']
Combined Score: 83.70884658454648
────────────────────────────────────────────────────────────────────────────────
Book:          TRANSCENDENT KINGDOM
Ebook:         TRANSCENDENT KINGDOM: A NOVEL
Book Author:   GYASI, YAA
Ebook Author:  YAA GYASI
Title Score:   81.6  |  Author

In [19]:
### check unmatched results 
unmatched = fuzzy_crossed_v6[fuzzy_crossed_v6['ebook_rank'].isna()].drop_duplicates('book_title')

candidates_list = []

#reimport clean_author and get_meaningful_words functions

min_substring_len=3
noise = {'a', 'an', 'the', 'of', 'in', 'and', 'or', 'to', 'by', 'for',
             'novel', 'memoir', 'story', 'book', 'vol', 'volume'}


def clean_author(author):
        author = re.sub(r'\(.*?\)', '', str(author))
        author = re.sub(r',?\s*(JR|SR|II|III|IV)\.?', '', author, flags=re.IGNORECASE)
        author = re.sub(r',?\s*\d{4}-?(\d{4})?', '', author)
        return author.strip()

def get_meaningful_words(title, author=None):
        words = set(re.sub(r'[^a-z0-9\s]', ' ', str(title).lower()).split())
        author_words = set(re.sub(r'[^a-z0-9\s]', ' ', str(author).lower()).split()) if author else set()
        return [w for w in words 
                if w not in noise 
                and len(w) >= min_substring_len
                and w not in author_words]

for _, row in unmatched.iterrows():
    year  = row['checkoutyear']
    month = row['checkoutmonth']
    
    ebook_month   = ebook_ranked[(ebook_ranked['checkoutyear'] == year) &
                                  (ebook_ranked['checkoutmonth'] == month)].drop_duplicates('title')
    ebook_authors = ebook_month['creator'].tolist()
    
    # Clean author for matching — same logic as in fuzzy_cross_rank_v2
    book_author_clean = clean_author(row['book_author'])
    book_words        = get_meaningful_words(row['book_title'], row['book_author'])
    min_words_required = max(1, len(book_words) * 0.75)

    # Step 1 — get passing authors
    author_candidates = process.extract(book_author_clean, ebook_authors,
                                         scorer=fuzz.token_sort_ratio, limit=10)
    passing_authors = [(score, idx) for _, score, idx in author_candidates
                       if score >= 70]

    # Step 2 — score titles among passing authors
    title_candidates = []
    for author_score, idx in passing_authors:
        candidate     = ebook_month.iloc[idx]
        title_score   = fuzz.WRatio(row['book_title'], candidate['title'])
        ebook_lower   = candidate['title'].lower()
        matched_words = [w for w in book_words if w in ebook_lower]
        combined      = (title_score * 0.6) + (author_score * 0.4)
        title_candidates.append({
            'book_title':           row['book_title'],
            'book_author':          row['book_author'],
            'ebook_title':          candidate['title'],
            'ebook_author':         candidate['creator'],
            'title_score':          round(title_score, 1),
            'author_score':         round(author_score, 1),
            'combined_score':       round(combined, 1),
            'matched_words':        matched_words,
            'num_matched_words':    len(matched_words),
            'min_words_required':   min_words_required,
            'passed_word_filter':   len(matched_words) >= min_words_required,
        })

    # Sort by combined score descending
    title_candidates.sort(key=lambda x: x['combined_score'], reverse=True)
    
    # Take top 5
    for rank, c in enumerate(title_candidates[:5], start=1):
        c['ebook_candidate_rank'] = rank
        candidates_list.append(c)

candidates_df = pd.DataFrame(candidates_list)

# Get top 20 unmatched books by best combined score
top20_books = (candidates_df.sort_values('combined_score', ascending=False)
               .drop_duplicates('book_title')
               .head(20)['book_title'].tolist())

print("=== Top 20 Unmatched Books (Method 6) — Top 5 Author-Matched Ebook Candidates ===\n")
for book_title in top20_books:
    book_candidates = candidates_df[candidates_df['book_title'] == book_title].sort_values('ebook_candidate_rank')
    best = book_candidates.iloc[0]
    min_req = best['min_words_required']
    print(f"Book:    {best['book_title'][:75]}")
    print(f"Author:  {best['book_author'][:50]}")
    print(f"Meaningful words: {get_meaningful_words(best['book_title'], best['book_author'])} → Min required: {min_req:.1f}")
    print(f"  {'Rank':<5} {'Title':>5} {'Author':>7} {'Combined':>9} {'Words':>6} {'Pass':>5}  Ebook Title | Ebook Author")
    for _, c in book_candidates.iterrows():
        passed = '✓' if c['passed_word_filter'] else '✗'
        print(f"  {c['ebook_candidate_rank']:<5} {c['title_score']:>5} {c['author_score']:>7} {c['combined_score']:>9} {c['num_matched_words']:>3}/{min_req:<3.0f} {passed:>5}  {c['ebook_title'][:50]} | {c['ebook_author'][:30]}")
    print()

=== Top 20 Unmatched Books (Method 6) — Top 5 Author-Matched Ebook Candidates ===

Book:    ATMOSPHERE : A LOVE STORY
Author:  REID, TAYLOR JENKINS
Meaningful words: ['love', 'atmosphere'] → Min required: 1.5
  Rank  Title  Author  Combined  Words  Pass  Ebook Title | Ebook Author
  1      90.0    97.4      93.0   1/2       ✗  ATMOSPHERE | TAYLOR JENKINS REID
  2      85.5    97.4      90.3   0/2       ✗  THE SEVEN HUSBANDS OF EVELYN HUGO: A NOVEL | TAYLOR JENKINS REID
  3      85.5    97.4      90.3   0/2       ✗  MALIBU RISING: A READ WITH JENNA PICK: A NOVEL | TAYLOR JENKINS REID
  4      54.5    97.4      71.7   0/2       ✗  AFTER I DO: A NOVEL | TAYLOR JENKINS REID
  5      50.0    97.4      69.0   1/2       ✗  ONE TRUE LOVES: A NOVEL | TAYLOR JENKINS REID

Book:    THE LET THEM THEORY : A LIFE-CHANGING TOOL THAT MILLIONS OF PEOPLE CAN'T ST
Author:  ROBBINS, MEL
Meaningful words: ['them', 'changing', 'life', 'can', 'stop', 'let', 'tool', 'millions', 'theory', 'talking', 'about', '